In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
dataset = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", dataset.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
import torch
import torch.nn as nn
import nltk
nltk.download("punkt_tab")
import re
from nltk.tokenize import word_tokenize



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
X = dataset["review"]
y = dataset["sentiment"]

In [ ]:
y.unique()
y = y.map({"positive" : 1, "negative" : 0})

In [ ]:
def preprocessing(text):
  lower = text.lower()
  base = re.sub(r'[^a-zA-Z0-9]\s', '', lower)
  tokenization = word_tokenize(base)

  return tokenization


In [ ]:
X = X.apply(preprocessing)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


tokenizer = Tokenizer(oov_token="<UNK>", num_words = 30000)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(X_train, padding = "post")
X_test =  tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(X_test, maxlen = X_train.shape[1],  padding = "post")

vocab = tokenizer.word_index


In [ ]:
Max_len = 256

X_train = pad_sequences(X_train, maxlen = Max_len, padding = "post", truncating = "post")
X_test = pad_sequences(X_test, maxlen = Max_len, padding = "post", truncating = "post")

In [ ]:

print(max(max(seq) for seq in X_train))

29999


In [ ]:
X_train_tensor = torch.tensor(X_train, dtype = torch.long)
X_test_tensor = torch.tensor(X_test, dtype = torch.long)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype = torch.long)

In [ ]:
X_train_tensor

tensor([[   13,    15,    58,  ...,    36,   148, 19794],
        [   17,    78,    31,  ...,   977,  7458,    22],
        [    3,  1412,   126,  ...,     0,     0,     0],
        ...,
        [  962,    16,  2938,  ...,     0,     0,     0],
        [   16,   382,    18,  ...,     0,     0,     0],
        [   16,    11,     3,  ...,     0,     0,     0]])

In [ ]:
# embedding = nn.Embedding(num_embeddings = len(vocab) + 1, embedding_dim = 128)

# X_train_embedded = embedding(X_train_tensor)
# X_test_embedded = embedding(X_test_tensor)

# print(X_train_embedded.shape)
# print(X_test_embedded.shape)

In [ ]:
print("Vocabulary Size:", len(vocab))
print("Maximum Sequence Length:", X_train_tensor.shape[1])
print("Number of Classes:", len(set(y_train)))
print("y_train dtype:", y_train_tensor.dtype)

Vocabulary Size: 549391
Maximum Sequence Length: 256
Number of Classes: 2
y_train dtype: torch.int64


In [ ]:
class Transformer(nn.Module):

  def __init__(self, embedding_dim = 128, max_length = 256, num_heads = 8, num_layers = 4, num_classes = 2, dropout = 0.1, dim_feedforward = 512):

    super().__init__()

    self.embedding = nn.Embedding(len(vocab), embedding_dim)

    self.positional_embedding = nn.Embedding(max_length, embedding_dim)

    encoder_layer = nn.TransformerEncoderLayer(d_model = embedding_dim, nhead = num_heads, dropout = dropout, dim_feedforward = dim_feedforward, batch_first = True)

    self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

    self.dropout = nn.Dropout(dropout)

    self.fc = nn.Linear(embedding_dim, num_classes)

  def forward(self, x):

    batch_size, seq_len = x.shape
    x = self.embedding(x)
    positions = torch.arange(seq_len, device = x.device).unsqueeze(0).expand(batch_size, seq_len)

    # print("x shape:", x.shape)
    # print("seq_len:", seq_len)
    # print("max_length:", self.positional_embedding.num_embeddings)

    x = x + self.positional_embedding(positions)

    x = self.encoder(x)

    x = x.mean(dim = 1)

    x = self.dropout(x)

    return self.fc(x)


In [ ]:
# Dataset and Data Loader

batch_size = 32

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size, shuffle = False)

In [ ]:
for X_batch, y_batch in train_loader:

    print(X_batch.shape)
    print(y_batch.shape)

    break

torch.Size([32, 256])
torch.Size([32])


In [ ]:
print(X_train_tensor.shape)

torch.Size([40000, 256])


In [ ]:
model = Transformer()
# outputs = model(X_train_tensor)

In [ ]:
print(torch.cuda.is_available())
print(device)
print(next(model.parameters()).device)

True
cuda
cuda:0


In [ ]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr = 0.0001)
criterion = nn.CrossEntropyLoss()


In [ ]:

epochs = 20

for epoch in range(epochs):


    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, dim=1)

        correct += (predicted == y_batch).sum().item()

        total += y_batch.size(0)

    accuracy = correct / total

    print(f"Epoch {epoch+1}")
    print(f"Loss: {total_loss / len(train_loader):.4f}")
    print(f"Accuracy: {accuracy:.4f}")

Epoch 1
Loss: 0.6341
Accuracy: 0.6368
Epoch 2
Loss: 0.6317
Accuracy: 0.6403
Epoch 3
Loss: 0.6307
Accuracy: 0.6432
Epoch 4
Loss: 0.6304
Accuracy: 0.6418
Epoch 5
Loss: 0.6304
Accuracy: 0.6445
Epoch 6
Loss: 0.6286
Accuracy: 0.6460
Epoch 7
Loss: 0.6297
Accuracy: 0.6452
Epoch 8
Loss: 0.6317
Accuracy: 0.6422
Epoch 9
Loss: 0.6317
Accuracy: 0.6431
Epoch 10
Loss: 0.6320
Accuracy: 0.6406
Epoch 11
Loss: 0.6329
Accuracy: 0.6418
Epoch 12
Loss: 0.6325
Accuracy: 0.6420
Epoch 13
Loss: 0.6317
Accuracy: 0.6415
Epoch 14
Loss: 0.6295
Accuracy: 0.6446
Epoch 15
Loss: 0.6250
Accuracy: 0.6514
Epoch 16
Loss: 0.6259
Accuracy: 0.6511
Epoch 17
Loss: 0.6263
Accuracy: 0.6491
Epoch 18
Loss: 0.6253
Accuracy: 0.6505
Epoch 19
Loss: 0.6255
Accuracy: 0.6513
Epoch 20
Loss: 0.6266
Accuracy: 0.6496


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

print(correct / total)

0.6237
